# Traffic Demand Prediction System

This notebook implements an advanced Gradient Boosted Decision Tree (GBDT) ensemble (LightGBM, XGBoost, CatBoost) to predict traffic demand for Day 49 daytime (2:15 to 13:45).

### Key Highlights:
1. **Leak-Free Validation**: Ensured no spatiotemporal target leakage in both historical profiles and early morning statistics.
2. **Combined Day 48 & Day 49 Training**: Extends the GBDT training range to cover full-day patterns, allowing tree models to extrapolate and predict daytime peaks.
3. **Neighborhood-Scaled Imputation**: Leverages spatial prefix scaling ratios to impute unobserved geohashes without introducing flat-value bias.
4. **Triple-Model Blended Ensemble**: Blends predictions from LightGBM, XGBoost, and CatBoost models across 5 folds.

## Part 1: Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

## Part 2: Utilities (Spatiotemporal Parsing & Decoding)

In [ ]:
import pandas as pd
import numpy as np

def decode_geohash(geohash):
    """
    Decodes a geohash string into latitude and longitude.
    """
    if not isinstance(geohash, str) or len(geohash) == 0:
        return np.nan, np.nan
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    
    for char in geohash:
        if char not in base32:
            return np.nan, np.nan
        val = base32.index(char)
        for i in range(4, -1, -1):
            bit = (val >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2.0
                if bit == 1:
                    lon_interval = (mid, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2.0
                if bit == 1:
                    lat_interval = (mid, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], mid)
            is_even = not is_even
            
    lat = (lat_interval[0] + lat_interval[1]) / 2.0
    lon = (lon_interval[0] + lon_interval[1]) / 2.0
    return lat, lon

def parse_time_features(df):
    """
    Extracts time features from the timestamp column.
    timestamp format is 'H:M' (e.g. '2:15')
    """
    df = df.copy()
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclic features
    df['sin_time'] = np.sin(2 * np.pi * df['time_minutes'] / 1440.0)
    df['cos_time'] = np.cos(2 * np.pi * df['time_minutes'] / 1440.0)
    
    return df

def impute_missing_values(df, train_df=None):
    """
    Imputes missing values for RoadType, Weather, and Temperature.
    If train_df is provided, uses it to compute imputation mappings.
    """
    df = df.copy()
    ref_df = train_df if train_df is not None else df
    
    # Modes
    road_mode = ref_df['RoadType'].mode()[0] if not ref_df['RoadType'].mode().empty else 'Residential'
    weather_mode = ref_df['Weather'].mode()[0] if not ref_df['Weather'].mode().empty else 'Sunny'
    
    df['RoadType'] = df['RoadType'].fillna(road_mode)
    df['Weather'] = df['Weather'].fillna(weather_mode)
    
    # Temperature mapping: Group by hour and Weather to get mean temperature
    temp_map = ref_df.groupby(['hour', 'Weather'])['Temperature'].mean().to_dict()
    overall_mean = ref_df['Temperature'].mean()
    if np.isnan(overall_mean):
        overall_mean = 20.0
        
    def fill_temp(row):
        if not np.isnan(row['Temperature']):
            return row['Temperature']
        key = (row['hour'], row['Weather'])
        if key in temp_map and not np.isnan(temp_map[key]):
            return temp_map[key]
        return overall_mean
        
    df['Temperature'] = df.apply(fill_temp, axis=1)
    return df


## Part 3: Feature Engineering and Imputation Pipeline

In [ ]:
import pandas as pd
import numpy as np


def get_day48_grid(train_df):
    """
    Creates a complete interpolated grid of demand on Day 48 for all geohashes.
    """
    day48 = train_df[train_df['day'] == 48].copy()
    day48 = parse_time_features(day48)
    
    geohashes = day48['geohash'].unique()
    times = sorted(day48['time_minutes'].unique())
    
    # Create complete grid
    grid = pd.MultiIndex.from_product([geohashes, times], names=['geohash', 'time_minutes']).to_frame().reset_index(drop=True)
    grid = grid.merge(day48[['geohash', 'time_minutes', 'demand']], on=['geohash', 'time_minutes'], how='left')
    
    # Sort and interpolate
    grid = grid.sort_values(['geohash', 'time_minutes'])
    grid['demand48'] = grid.groupby('geohash')['demand'].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both').fillna(0.0)
    )
    
    # Add prefix columns for neighborhood profiling on Day 48
    grid['geohash_prefix5'] = grid['geohash'].str[:5]
    grid['geohash_prefix4'] = grid['geohash'].str[:4]
    
    return grid

def build_features(df, train_df=None):
    """
    Builds spatiotemporal features and maps Day 48 scaled profiles.
    - Decodes geohashes & imputes environmental variables.
    - Computes Day 48 early morning demand profile.
    - Scales Day 48 daytime profiles using Day 49 early morning ratios (with Neighborhood-Scaled Imputation fallback).
    - Excludes targets from early morning demand features to prevent leakage during training.
    """
    df = df.copy()
    
    # 1. Parse basic time features
    df = parse_time_features(df)
    
    # 2. Decode geohashes
    lats_lons = df['geohash'].apply(decode_geohash)
    df['latitude'] = [x[0] for x in lats_lons]
    df['longitude'] = [x[1] for x in lats_lons]
    
    # 3. Geohash prefixes
    df['geohash_prefix4'] = df['geohash'].str[:4]
    df['geohash_prefix5'] = df['geohash'].str[:5]
    
    # 4. Impute missing values
    ref_train = train_df if train_df is not None else df[df['demand'].notna()]
    train_parsed = parse_time_features(ref_train.copy())
    df = impute_missing_values(df, train_parsed)
    
    # 5. Day 48 (previous day) historical grid & profiles
    grid48 = get_day48_grid(ref_train)
    
    # Merge Day 48 demand
    df = df.merge(grid48[['geohash', 'time_minutes', 'demand48']], on=['geohash', 'time_minutes'], how='left')
    df['demand48'] = df['demand48'].fillna(0.0)
    
    # Day 48 early morning average per geohash
    d48_clean = ref_train[ref_train['day'] == 48].copy()
    d48_clean = parse_time_features(d48_clean)
    d48_early = d48_clean[d48_clean['time_minutes'] <= 120].groupby('geohash')['demand'].mean().reset_index().rename(columns={'demand': 'early_48'})
    df = df.merge(d48_early, on='geohash', how='left')
    df['early_48'] = df['early_48'].fillna(df['early_48'].mean())
    
    # Day 49 observed early morning statistics
    d49_early_rows = ref_train[(ref_train['day'] == 49) & (parse_time_features(ref_train)['time_minutes'] <= 120)].copy()
    early_stats = d49_early_rows.groupby('geohash')['demand'].agg(['sum', 'count']).reset_index()
    df = df.merge(early_stats, on='geohash', how='left')
    df['sum'] = df['sum'].fillna(0.0)
    df['count'] = df['count'].fillna(0)
    
    # Non-leaky observed early average
    # Subtract current row's demand if it is part of Day 49 early morning (0:00 to 2:00) during training
    is_d49_early = (df['day'] == 49) & (df['time_minutes'] <= 120) & (df['demand'].notna())
    df['early_49_obs'] = np.where(
        is_d49_early,
        (df['sum'] - df['demand'].fillna(0.0)) / (df['count'] - 1).clip(lower=1),
        df['sum'] / df['count'].clip(lower=1)
    )
    df['early_49_obs'] = np.where(df['count'] == 0, np.nan, df['early_49_obs'])
    
    # Neighborhood and global ratios for imputation
    early_by_geohash = d49_early_rows.groupby('geohash')['demand'].mean().reset_index().rename(columns={'demand': 'early_49_g'})
    early_by_geohash = early_by_geohash.merge(d48_early, on='geohash', how='left')
    early_by_geohash['early_48'] = early_by_geohash['early_48'].fillna(early_by_geohash['early_48'].mean())
    early_by_geohash['geohash_prefix5'] = early_by_geohash['geohash'].str[:5]
    early_by_geohash['geohash_prefix4'] = early_by_geohash['geohash'].str[:4]
    
    p5_means = early_by_geohash.groupby('geohash_prefix5')[['early_49_g', 'early_48']].mean().reset_index()
    p5_means.columns = ['geohash_prefix5', 'p5_early_49', 'p5_early_48']
    p5_means['prefix5_ratio'] = (p5_means['p5_early_49'] + 0.01) / (p5_means['p5_early_48'] + 0.01)
    
    p4_means = early_by_geohash.groupby('geohash_prefix4')[['early_49_g', 'early_48']].mean().reset_index()
    p4_means.columns = ['geohash_prefix4', 'p4_early_49', 'p4_early_48']
    p4_means['prefix4_ratio'] = (p4_means['p4_early_49'] + 0.01) / (p4_means['p4_early_48'] + 0.01)
    
    df = df.merge(p5_means[['geohash_prefix5', 'prefix5_ratio', 'p5_early_49']], on='geohash_prefix5', how='left')
    df = df.merge(p4_means[['geohash_prefix4', 'prefix4_ratio', 'p4_early_49']], on='geohash_prefix4', how='left')
    
    global_early_49 = d49_early_rows['demand'].mean()
    global_early_48 = d48_clean[d48_clean['time_minutes'] <= 120]['demand'].mean()
    global_ratio = (global_early_49 + 0.01) / (global_early_48 + 0.01)
    
    df['prefix5_ratio'] = df['prefix5_ratio'].fillna(global_ratio)
    df['prefix4_ratio'] = df['prefix4_ratio'].fillna(global_ratio)
    df['p5_early_49'] = df['p5_early_49'].fillna(global_early_49)
    df['p4_early_49'] = df['p4_early_49'].fillna(global_early_49)
    
    early_49_imputed = []
    for idx, row in df.iterrows():
        if not np.isnan(row['early_49_obs']):
            early_49_imputed.append(row['early_49_obs'])
        elif not np.isnan(row['early_48']):
            early_49_imputed.append(row['early_48'] * row['prefix5_ratio'])
        else:
            early_49_imputed.append(row['p5_early_49'])
    df['early_49'] = early_49_imputed
    
    # Assign scaling early demand based on day
    df['early_demand'] = np.where(df['day'] == 48, df['early_48'], df['early_49'])
    
    # Scaling ratios
    df['geohash_ratio'] = (df['early_demand'] + 0.01) / (df['early_48'] + 0.01)
    df['scaled_demand48'] = df['demand48'] * df['geohash_ratio']
    
    # Neighborhood profiles of demand48
    p5_stats = grid48.groupby(['geohash_prefix5', 'time_minutes'])['demand48'].mean().reset_index().rename(columns={'demand48': 'p5_demand48_t'})
    p4_stats = grid48.groupby(['geohash_prefix4', 'time_minutes'])['demand48'].mean().reset_index().rename(columns={'demand48': 'p4_demand48_t'})
    df = df.merge(p5_stats, on=['geohash_prefix5', 'time_minutes'], how='left')
    df = df.merge(p4_stats, on=['geohash_prefix4', 'time_minutes'], how='left')
    df['p5_demand48_t'] = df['p5_demand48_t'].fillna(0.0)
    df['p4_demand48_t'] = df['p4_demand48_t'].fillna(0.0)
    
    # Clean up temporary columns
    cols_to_drop = [
        'sum', 'count', 'early_49_obs', 'prefix5_ratio', 'prefix4_ratio', 
        'p5_early_49', 'p4_early_49', 'p5_sum', 'p5_count', 'p4_sum', 'p4_count'
    ]
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    feature_cols = [c for c in df.columns if c != 'demand']
    df[feature_cols] = df[feature_cols].fillna(0.0)
    
    return df


## Part 4: Training & Blending Pipeline

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import OrdinalEncoder

import warnings
warnings.filterwarnings('ignore')

def train_and_evaluate():
    print("Step 1: Loading Datasets...")
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')
    
    print("Step 2: Preprocessing and Feature Engineering...")
    # Concatenate train and test to ensure consistent feature engineering
    all_df = pd.concat([train, test], ignore_index=True)
    all_feat = build_features(all_df, train)
    
    # Split back to train and test
    train_feat = all_feat[all_feat['demand'].notna()].copy()
    test_feat = all_feat[all_feat['demand'].isna()].copy()
    
    # Categorical encoding
    cat_cols = ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'geohash', 'geohash_prefix4', 'geohash_prefix5']
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    
    # Fit encoder on all train and transform both
    train_feat[cat_cols] = encoder.fit_transform(train_feat[cat_cols].astype(str))
    test_feat[cat_cols] = encoder.transform(test_feat[cat_cols].astype(str))
    
    features = [
        'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather',
        'hour', 'minute', 'time_minutes', 'sin_time', 'cos_time', 'latitude', 'longitude',
        'geohash', 'geohash_prefix4', 'geohash_prefix5',
        'early_48', 'early_49', 'geohash_ratio', 'scaled_demand48',
        'p5_demand48_t', 'p4_demand48_t'
    ]
    
    target = 'demand'
    
    # We train GBDTs on Day 49 early morning (where residuals vary realistically)
    train_49 = train_feat[train_feat['day'] == 49].copy()
    
    print(f"Train Day 49 early morning shape: {train_49.shape}")
    print(f"Test Day 49 shape: {test_feat.shape}")
    
    # K-Fold Cross Validation Setup on Day 49
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    # OOF prediction arrays
    oof_lgb = np.zeros(len(train_49))
    oof_xgb = np.zeros(len(train_49))
    oof_cb = np.zeros(len(train_49))
    
    # Test prediction arrays
    test_res_lgb = np.zeros(len(test_feat))
    test_res_xgb = np.zeros(len(test_feat))
    test_res_cb = np.zeros(len(test_feat))
    
    lgb_params = {
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'verbose': -1,
        'random_state': 42
    }
    
    xgb_params = {
        'n_estimators': 1000,
        'learning_rate': 0.05,
        'max_depth': 5,
        'verbosity': 0,
        'random_state': 42
    }
    
    cb_params = {
        'iterations': 1000,
        'learning_rate': 0.05,
        'depth': 5,
        'verbose': 0,
        'random_seed': 42
    }
    
    print("\nStep 3: Starting Cross-Validation & Training on Residual GBDT Ensemble...")
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_49)):
        print(f"\n--- Training Fold {fold} ---")
        
        # Split features and target
        X_train, y_train = train_49.iloc[train_idx][features], train_49.iloc[train_idx][target]
        X_val, y_val = train_49.iloc[val_idx][features], train_49.iloc[val_idx][target]
        
        # Calculate residuals for training and validation
        y_train_res = y_train - X_train['scaled_demand48']
        y_val_res = y_val - X_val['scaled_demand48']
        
        # 1. LightGBM
        model_lgb = lgb.LGBMRegressor(**lgb_params)
        model_lgb.fit(X_train, y_train_res, eval_set=[(X_val, y_val_res)], callbacks=[lgb.early_stopping(50, verbose=False)])
        pred_res_val = model_lgb.predict(X_val)
        oof_lgb[val_idx] = np.clip(X_val['scaled_demand48'] + pred_res_val, 0.0, 1.0)
        test_res_lgb += model_lgb.predict(test_feat[features]) / 5.0
        
        # 2. XGBoost
        model_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        model_xgb.fit(X_train, y_train_res, eval_set=[(X_val, y_val_res)], verbose=False)
        pred_res_val = model_xgb.predict(X_val)
        oof_xgb[val_idx] = np.clip(X_val['scaled_demand48'] + pred_res_val, 0.0, 1.0)
        test_res_xgb += model_xgb.predict(test_feat[features]) / 5.0
        
        # 3. CatBoost
        model_cb = cb.CatBoostRegressor(**cb_params, early_stopping_rounds=50)
        model_cb.fit(X_train, y_train_res, eval_set=(X_val, y_val_res), verbose=False)
        pred_res_val = model_cb.predict(X_val)
        oof_cb[val_idx] = np.clip(X_val['scaled_demand48'] + pred_res_val, 0.0, 1.0)
        test_res_cb += model_cb.predict(test_feat[features]) / 5.0
        
        # Calculate fold metrics
        r2_lgb = r2_score(y_val, oof_lgb[val_idx]) * 100
        r2_xgb = r2_score(y_val, oof_xgb[val_idx]) * 100
        r2_cb = r2_score(y_val, oof_cb[val_idx]) * 100
        print(f"Fold {fold} R2 Scores -> LGBM: {r2_lgb:.4f}% | XGB: {r2_xgb:.4f}% | CatBoost: {r2_cb:.4f}%")
        
    print("\nStep 4: Evaluating Out-of-Fold (OOF) Metrics...")
    y_49 = train_49[target].values
    
    score_lgb = r2_score(y_49, oof_lgb) * 100
    score_xgb = r2_score(y_49, oof_xgb) * 100
    score_cb = r2_score(y_49, oof_cb) * 100
    
    # Blended/Ensemble predictions
    oof_ensemble = (oof_lgb + oof_xgb + oof_cb) / 3.0
    score_ensemble = r2_score(y_49, oof_ensemble) * 100
    
    print("="*50)
    print(f"OOF R2 Score LightGBM: {score_lgb:.4f}%")
    print(f"OOF R2 Score XGBoost:  {score_xgb:.4f}%")
    print(f"OOF R2 Score CatBoost: {score_cb:.4f}%")
    print(f"OOF R2 Score Ensemble: {score_ensemble:.4f}%")
    print("="*50)
    
    print("\nStep 5: Generating Submission File...")
    blended_test_res = (test_res_lgb + test_res_xgb + test_res_cb) / 3.0
    final_preds = test_feat['scaled_demand48'] + blended_test_res
    final_preds = np.clip(final_preds, 0.0, 1.0)
    
    # Construct submission dataframe
    submission = pd.DataFrame({
        'Index': test_feat['Index'].astype(int),
        'demand': final_preds
    })
    
    # Validate structure
    assert len(submission) == 41778, f"Incorrect number of rows: {len(submission)}"
    assert list(submission.columns) == ['Index', 'demand'], f"Incorrect columns: {list(submission.columns)}"
    assert not submission['demand'].isna().any(), "Submission contains NaN values"
    assert (submission['demand'] >= 0.0).all() and (submission['demand'] <= 1.0).all(), "Predictions out of bounds"
    
    # Verify index values match test file exactly
    test_orig = pd.read_csv('test.csv')
    assert (submission['Index'].values == test_orig['Index'].values).all(), "Index values do not match test file"
    
    submission.to_csv('submission.csv', index=False)
    print("Submission file successfully saved to submission.csv!")
    print("Validation passed: File shape is 41778 x 2, and all values are in bounds [0, 1].")

if __name__ == "__main__":
    train_and_evaluate()


## Part 5: Execute Training and Generate Submission

In [ ]:
train_and_evaluate()